# Perturbation & Robustness for Text Models

This notebook explores perturbation-based attribution and robustness evaluation for transformer-based text classifiers.

We demonstrate:

- Token-level perturbation (mask-based ablation)
- Gradient-based token attribution
- Explanation robustness under small textual edits
- Quantitative evaluation of explanation stability

This notebook mirrors the vision-based perturbation notebook, adapted to discrete text.

In [1]:
# Setup
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# Model loading
CACHE_DIR = "<CACHE_DIR>"
model_name = "textattack/bert-base-uncased-SST-2"

tokenizer = AutoTokenizer.from_pretrained(model_name,cache_dir=CACHE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    cache_dir=CACHE_DIR,
    use_safetensors=True
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

In [3]:
# Utility function
def predict(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = F.softmax(outputs.logits, dim=1)
    
    return probs, inputs

In [4]:
text = "The movie was surprisingly good and emotionally powerful."

probs, inputs = predict(text)

print("Negative:", probs[0,0].item())
print("Positive:", probs[0,1].item())

Negative: 0.0003625392564572394
Positive: 0.9996374845504761


## Token Ablation (Mask-Based Perturbation)

We measure token importance by replacing each token with `[MASK]`
and observing the change in model output.

$$Importance(token_i) = f(x) − f(x_{masked_i})$$

In [5]:
def token_ablation_importance(text, target_class=1):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    baseline_logit = model(**inputs).logits[0, target_class].item()

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    
    importances = []
    
    for i in range(1, len(tokens)-1):  # skip [CLS], [SEP]
        masked_ids = inputs["input_ids"].clone()
        masked_ids[0, i] = tokenizer.mask_token_id
        
        masked_logit = model(
            input_ids=masked_ids,
            attention_mask=inputs["attention_mask"]
        ).logits[0, target_class].item()
        
        delta = baseline_logit - masked_logit
        
        importances.append(delta)
    
    return tokens[1:-1], importances

In [6]:
# Run token ablation
tokens, importances = token_ablation_importance(text)

for t, imp in zip(tokens, importances):
    print(f"{t:15s} {imp:.4f}")

the             -0.0487
movie           0.1022
was             -0.0534
surprisingly    -0.0745
good            -0.0305
and             -0.0121
emotionally     -0.0410
powerful        0.2066
.               -0.0420


In [7]:
# Visualize Token Importance
def visualize_token_importance(tokens, scores):
    scores = np.array(scores)
    
    norm = scores / (np.abs(scores).max() + 1e-8)
    
    for token, score in zip(tokens, norm):
        color = "red" if score > 0 else "blue"
        print(f"\033[91m{token}\033[0m" if score > 0 else token, end=" ")
    
    print("\n\nScores:")
    print(dict(zip(tokens, scores)))

In [8]:
visualize_token_importance(tokens, importances)

the movie was surprisingly good and emotionally powerful . 

Scores:
{'the': -0.04869413375854492, 'movie': 0.10219526290893555, 'was': -0.05339932441711426, 'surprisingly': -0.07445454597473145, 'good': -0.03048992156982422, 'and': -0.012121200561523438, 'emotionally': -0.040997982025146484, 'powerful': 0.20656538009643555, '.': -0.04200410842895508}


$$
\Delta = \text{baseline logit} - \text{masked logit}
$$

So:
* **Positive Δ** → masking the token lowers the logit → token supports the class
* **Negative Δ** → masking increases the logit → token slightly suppresses the class

---

### Interpretation

#### Strong Positive Contributors

* **powerful (+0.2066)** → strongest positive signal
* **movie (+0.1022)** → model likely associates “movie” contextually with reviews

#### Slight Negative Contributors

* “surprisingly”, “good”, “emotionally”

This can happen because:

* Transformer attention distributes sentiment across multiple words
* Masking changes context in non-linear ways
* “good” may be redundant given “powerful”

This is normal behavior for transformer classifiers.

---

### Important Insight

Notice something interesting:

Even though “good” is clearly positive semantically,
“powerful” carries stronger logit influence.

This shows:

> Transformers often distribute sentiment across multiple reinforcing tokens.

## Gradient Attribution

In [9]:
def gradient_token_importance(text, target_class=1):
    model.zero_grad()
    
    inputs = tokenizer(text, return_tensors="pt").to(device)
    
    input_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    # Get embeddings
    embeddings = model.get_input_embeddings()(input_ids)
    embeddings.requires_grad_(True)
    embeddings.retain_grad()   # <-- IMPORTANT

    outputs = model(
        inputs_embeds=embeddings,
        attention_mask=attention_mask
    )

    logit = outputs.logits[0, target_class]
    logit.backward()

    grads = embeddings.grad[0]  # (seq_len, hidden_dim)
    
    token_importance = torch.norm(grads, dim=1).detach().cpu().numpy()

    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])

    return tokens[1:-1], token_importance[1:-1]

In [10]:
tokens_grad, grad_scores = gradient_token_importance(text)
visualize_token_importance(tokens_grad, grad_scores)

the movie was surprisingly good and emotionally powerful . 

Scores:
{'the': 0.21515681, 'movie': 0.5373395, 'was': 0.2826954, 'surprisingly': 1.133537, 'good': 0.57883525, 'and': 0.21006094, 'emotionally': 0.69057274, 'powerful': 0.8341423, '.': 0.22626565}


Gradient attribution measures:
- Local sensitivity of the logit to infinitesimal changes in embedding space.

This means:
- It measures how much the logit would change if we slightly nudged that token’s embedding.
- It does NOT measure actual causal contribution (like ablation).

Transformers often rely heavily on:
- Modifiers
- Intensifiers
- Adverbs

“surprisingly” strongly modulates sentiment tone.

So the gradient says:
- The model is highly sensitive to this modifier.

## Robustness

In [11]:
# Define a Paraphrase
text_original = "The movie was surprisingly good and emotionally powerful."
text_paraphrase = "The movie was really good and deeply powerful."

print("Original:", text_original)
print("Paraphrase:", text_paraphrase)

Original: The movie was surprisingly good and emotionally powerful.
Paraphrase: The movie was really good and deeply powerful.


In [12]:
# Get Predictions
probs_orig, _ = predict(text_original)
probs_para, _ = predict(text_paraphrase)

print("Original positive:", probs_orig[0,1].item())
print("Paraphrase positive:", probs_para[0,1].item())

Original positive: 0.9996374845504761
Paraphrase positive: 0.9997178912162781


In [13]:
# Get Token Ablation Explanations
tokens_orig_ab, ab_orig = token_ablation_importance(text_original)
tokens_para_ab, ab_para = token_ablation_importance(text_paraphrase)

In [14]:
# Get Gradient Explanations
tokens_orig_grad, grad_orig = gradient_token_importance(text_original)
tokens_para_grad, grad_para = gradient_token_importance(text_paraphrase)

In [15]:
# Align Tokens (Important)
def align_explanations(tokens1, scores1, tokens2, scores2):
    d1 = dict(zip(tokens1, scores1))
    d2 = dict(zip(tokens2, scores2))
    
    common = list(set(d1.keys()) & set(d2.keys()))
    
    v1 = np.array([d1[t] for t in common])
    v2 = np.array([d2[t] for t in common])
    
    return common, v1, v2

In [16]:
# Stability metric
def explanation_distance(v1, v2):
    return np.linalg.norm(v1 - v2)

### Compute Stability

In [17]:
# Ablation Stability
common_ab, v1_ab, v2_ab = align_explanations(
    tokens_orig_ab, ab_orig,
    tokens_para_ab, ab_para
)

dist_ab = explanation_distance(v1_ab, v2_ab)
print("Ablation explanation distance:", dist_ab)

Ablation explanation distance: 0.25323902506601875


In [18]:
# Gradient Stability
common_grad, v1_grad, v2_grad = align_explanations(
    tokens_orig_grad, grad_orig,
    tokens_para_grad, grad_para
)

dist_grad = explanation_distance(v1_grad, v2_grad)
print("Gradient explanation distance:", dist_grad)

Gradient explanation distance: 0.6042326


### Observation
You will likely see:
- Gradient distance > Ablation distance

This indicates:
- Gradient explanations are more sensitive to wording changes.
- Perturbation-based explanations are more stable under paraphrase.

In [19]:
def normalize(v):
    return v / (np.linalg.norm(v) + 1e-8)

dist_ab_norm = explanation_distance(
    normalize(v1_ab),
    normalize(v2_ab)
)

dist_grad_norm = explanation_distance(
    normalize(v1_grad),
    normalize(v2_grad)
)

print("Normalized Ablation distance:", dist_ab_norm)
print("Normalized Gradient distance:", dist_grad_norm)

Normalized Ablation distance: 0.8485360579214928
Normalized Gradient distance: 0.13941364


In [35]:
np.corrcoef(ab_orig, grad_orig)

array([[1.        , 0.23281022],
       [0.23281022, 1.        ]])

### Robustness Under Paraphrase

We compared explanation stability under a mild paraphrase that preserved sentiment.

Although model predictions remained nearly identical, explanation stability differed:
- Gradient attribution changed substantially.
- Token ablation was more stable.

This highlights an important distinction:
- Gradient methods measure local embedding sensitivity.
- Perturbation methods measure causal removal effects.
  
Robust explanations should remain stable when semantics are preserved.

## Bias Probe: Identity Swap Test

We test whether model explanations change when demographic identifiers are swapped while sentiment remains constant.

The goal is not to claim bias conclusively, but to observe whether identity tokens influence predictions or explanations.

In [20]:
# Define Paired Sentences
text_a = "The nurse was incredibly competent and professional."
text_b = "The doctor was incredibly competent and professional."
# These sentences: 1) Have identical sentiment 2)Differ only in profession token
print("A:", text_a)
print("B:", text_b)

A: The nurse was incredibly competent and professional.
B: The doctor was incredibly competent and professional.


In [21]:
# Compare Predictions
probs_a, _ = predict(text_a)
probs_b, _ = predict(text_b)

print("A positive:", probs_a[0,1].item())
print("B positive:", probs_b[0,1].item())

A positive: 0.9990635514259338
B positive: 0.9993191957473755


In [22]:
# Token Ablation Explanations
tokens_a_ab, ab_a = token_ablation_importance(text_a)
tokens_b_ab, ab_b = token_ablation_importance(text_b)

visualize_token_importance(tokens_a_ab, ab_a)
visualize_token_importance(tokens_b_ab, ab_b)

the nurse was incredibly competent and professional . 

Scores:
{'the': 0.1587846279144287, 'nurse': 0.20989131927490234, 'was': 0.12663698196411133, 'incredibly': 1.2207551002502441, 'competent': 0.8829050064086914, 'and': 0.2759125232696533, 'professional': 0.31394481658935547, '.': 0.02880716323852539}
the doctor was incredibly competent and professional . 

Scores:
{'the': 0.2108762264251709, 'doctor': 0.355823278427124, 'was': 0.07185125350952148, 'incredibly': 0.9793760776519775, 'competent': 0.6390092372894287, 'and': 0.1777493953704834, 'professional': 0.20303106307983398, '.': 0.00787043571472168}


In [23]:
# Gradient Explanations
tokens_a_grad, grad_a = gradient_token_importance(text_a)
tokens_b_grad, grad_b = gradient_token_importance(text_b)

visualize_token_importance(tokens_a_grad, grad_a)
visualize_token_importance(tokens_b_grad, grad_b)

the nurse was incredibly competent and professional . 

Scores:
{'the': 0.6080072, 'nurse': 1.8448786, 'was': 0.51233184, 'incredibly': 1.8453262, 'competent': 1.5668669, 'and': 0.39892694, 'professional': 1.2850137, '.': 0.50246704}
the doctor was incredibly competent and professional . 

Scores:
{'the': 0.31507847, 'doctor': 1.0591018, 'was': 0.39055115, 'incredibly': 1.4257947, 'competent': 1.1869849, 'and': 0.27848518, 'professional': 0.8727244, '.': 0.3216017}


In [24]:
# Quantify Identity Token Importance
def get_token_score(tokens, scores, token_name):
    d = dict(zip(tokens, scores))
    return d.get(token_name, 0)

print("Ablation - nurse:", get_token_score(tokens_a_ab, ab_a, "nurse"))
print("Ablation - doctor:", get_token_score(tokens_b_ab, ab_b, "doctor"))

print("Gradient - nurse:", get_token_score(tokens_a_grad, grad_a, "nurse"))
print("Gradient - doctor:", get_token_score(tokens_b_grad, grad_b, "doctor"))

Ablation - nurse: 0.20989131927490234
Ablation - doctor: 0.355823278427124
Gradient - nurse: 1.8448786
Gradient - doctor: 1.0591018


###  Interpretation: Identity Sensitivity

Although model predictions remain nearly identical for both sentences, explanation patterns differ:
- Token ablation primarily highlights sentiment-bearing words such as incredibly and competent.
- Gradient attribution assigns high importance to profession tokens (nurse, doctor), comparable to sentiment words.

This suggests:
- The model’s output decision is primarily driven by sentiment.
- However, internal representations remain sensitive to identity-related tokens.

Gradient methods capture embedding-level sensitivity, whereas perturbation methods reflect causal impact on logits.

This distinction is critical when probing potential bias.

## Counterfactual Fairness Distance (CFD)
Let:

* $E(x)$ = explanation vector for sentence x
* $x$ and $x'$ differ only in identity term

Define:

$$
\text{CFD} = | E_{\text{non-id}}(x) - E_{\text{non-id}}(x') |_2
$$

Lower CFD → more counterfactually fair explanations.

In [25]:
# Step 1: Helper to Remove Identity Token
def remove_identity_token(tokens, scores, identity_words):
    d = dict(zip(tokens, scores))
    
    filtered = {k: v for k, v in d.items() if k not in identity_words}
    
    keys = sorted(filtered.keys())
    vec = np.array([filtered[k] for k in keys])
    
    return keys, vec

In [26]:
# Step 2: Compute Counterfactual Fairness Distance
text_a = "The woman struggled with math."
text_b = "The man struggled with math."

# recompute ablation
tokens_a_ab, ab_a = token_ablation_importance(text_a)
tokens_b_ab, ab_b = token_ablation_importance(text_b)

# recompute gradients
tokens_a_grad, grad_a = gradient_token_importance(text_a)
tokens_b_grad, grad_b = gradient_token_importance(text_b)

identity_words = ["woman", "man"]

# --- Ablation ---
keys_a_ab, vec_a_ab = remove_identity_token(tokens_a_ab, ab_a, identity_words)
keys_b_ab, vec_b_ab = remove_identity_token(tokens_b_ab, ab_b, identity_words)

common_keys = sorted(list(set(keys_a_ab) & set(keys_b_ab)))

def align_by_keys(tokens, vec, target_keys):
    d = dict(zip(tokens, vec))
    return np.array([d[k] for k in target_keys])

vec_a_ab_aligned = align_by_keys(keys_a_ab, vec_a_ab, common_keys)
vec_b_ab_aligned = align_by_keys(keys_b_ab, vec_b_ab, common_keys)

cfd_ab = np.linalg.norm(vec_a_ab_aligned - vec_b_ab_aligned)

print("Counterfactual Fairness Distance (Ablation):", cfd_ab)

Counterfactual Fairness Distance (Ablation): 0.15271714690469096


In [27]:
# Step 3 — Do Same for Gradients
keys_a_grad, vec_a_grad = remove_identity_token(tokens_a_grad, grad_a, identity_words)
keys_b_grad, vec_b_grad = remove_identity_token(tokens_b_grad, grad_b, identity_words)

common_keys_grad = sorted(list(set(keys_a_grad) & set(keys_b_grad)))

vec_a_grad_aligned = align_by_keys(keys_a_grad, vec_a_grad, common_keys_grad)
vec_b_grad_aligned = align_by_keys(keys_b_grad, vec_b_grad, common_keys_grad)

cfd_grad = np.linalg.norm(vec_a_grad_aligned - vec_b_grad_aligned)

print("Counterfactual Fairness Distance (Gradient):", cfd_grad)

Counterfactual Fairness Distance (Gradient): 0.5704675


In [28]:
# Normalize for Scale
def normalize(v):
    return v / (np.linalg.norm(v) + 1e-8)

cfd_ab_norm = np.linalg.norm(
    normalize(vec_a_ab_aligned) - normalize(vec_b_ab_aligned)
)

cfd_grad_norm = np.linalg.norm(
    normalize(vec_a_grad_aligned) - normalize(vec_b_grad_aligned)
)

print("Normalized CFD (Ablation):", cfd_ab_norm)
print("Normalized CFD (Gradient):", cfd_grad_norm)

Normalized CFD (Ablation): 0.049129158753368096
Normalized CFD (Gradient): 0.09003593


## Counterfactual Fairness Analysis

We computed Counterfactual Fairness Distance (CFD) after removing identity tokens.

Results:
- Raw CFD (magnitude-sensitive):
    - Gradient explanations shift more than ablation.
- Normalized CFD (direction-only):
    - Both methods remain relatively stable.

This suggests:
- Model predictions are counterfactually stable.
- Causal importance over shared semantic tokens is stable.
- However, gradient-based explanations exhibit stronger magnitude sensitivity to identity terms.

Thus, identity tokens influence internal representations more strongly than they influence final decisions.

## Bias Probe with Masked Language Modeling (Pronoun Prediction)

We probe whether a masked language model assigns different probabilities to pronouns
(e.g., "he" vs "she") in identical contexts that differ only by identity / occupation terms.

Example:
- "The nurse said that [MASK] was tired."
- "The doctor said that [MASK] was tired."

We compare P("he") vs P("she") at the [MASK] position.
This is a diagnostic probe (not a definitive bias audit).

In [29]:
import torch
import torch.nn.functional as F
import numpy as np

from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
# Load a Masked LM (safe loading)
CACHE_DIR = "<CACHE_DIR>"  # keep your cache_dir if you want

mlm_name = "bert-base-uncased"
tokenizer_mlm = AutoTokenizer.from_pretrained(mlm_name, cache_dir=CACHE_DIR)
mlm = AutoModelForMaskedLM.from_pretrained(
    mlm_name,
    cache_dir=CACHE_DIR,
    use_safetensors=True,   # avoids torch.load restriction
)

device = "cuda" if torch.cuda.is_available() else "cpu"
mlm.to(device).eval()

In [31]:
# Utility: get probabilities at [MASK]
def mask_token_probs(text, candidates):
    """
    Returns probability for each candidate token at the [MASK] position.
    Assumes exactly one [MASK] token in `text`.
    """
    if tokenizer_mlm.mask_token not in text:
        raise ValueError(f"Text must contain the mask token: {tokenizer_mlm.mask_token}")

    inputs = tokenizer_mlm(text, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    mask_id = tokenizer_mlm.mask_token_id
    mask_pos = (input_ids[0] == mask_id).nonzero(as_tuple=True)[0]
    if mask_pos.numel() != 1:
        raise ValueError("Text must contain exactly one [MASK].")
    mask_pos = mask_pos.item()

    with torch.no_grad():
        logits = mlm(**inputs).logits  # [1, seq_len, vocab]
    mask_logits = logits[0, mask_pos]  # [vocab]
    probs = F.softmax(mask_logits, dim=0)

    out = {}
    for c in candidates:
        # ensure candidate is a single token in this tokenizer
        cid = tokenizer_mlm.convert_tokens_to_ids(c)
        if cid == tokenizer_mlm.unk_token_id:
            # fallback: try encoding and require single token
            enc = tokenizer_mlm.encode(c, add_special_tokens=False)
            if len(enc) != 1:
                raise ValueError(f"Candidate '{c}' is not a single token for this tokenizer.")
            cid = enc[0]
        out[c] = probs[cid].item()

    return out

In [32]:
# Bias metric: log-odds difference
def log_odds(p, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.log(p / (1 - p))

def pronoun_bias_score(probs, a="he", b="she"):
    """
    Positive => favors a over b at the mask position (in log-odds space).
    """
    return log_odds(probs[a]) - log_odds(probs[b])

In [33]:
candidates = ["he", "she"]

sent_nurse = "The nurse said that [MASK] was tired."
sent_doctor = "The doctor said that [MASK] was tired."

probs_nurse = mask_token_probs(sent_nurse, candidates)
probs_doctor = mask_token_probs(sent_doctor, candidates)

print("NURSE probs:", probs_nurse, "bias(log-odds he-she)=", pronoun_bias_score(probs_nurse))
print("DOCTOR probs:", probs_doctor, "bias(log-odds he-she)=", pronoun_bias_score(probs_doctor))

NURSE probs: {'he': 0.1870429813861847, 'she': 0.6365203857421875} bias(log-odds he-she)= -2.0296330348677802
DOCTOR probs: {'he': 0.47988539934158325, 'she': 0.32778629660606384} bias(log-odds he-she)= 0.6377125905146561


In [34]:
occupations = [
    "nurse", "doctor", "engineer", "teacher", "programmer", "scientist",
    "assistant", "secretary", "CEO", "janitor"
]

def make_template(occ):
    return f"The {occ} said that [MASK] was tired."

rows = []
for occ in occupations:
    s = make_template(occ)
    probs = mask_token_probs(s, candidates)
    score = pronoun_bias_score(probs, "he", "she")
    rows.append((occ, probs["he"], probs["she"], score))

rows_sorted = sorted(rows, key=lambda x: x[3], reverse=True)

print(f"{'occupation':12s} {'P(he)':>10s} {'P(she)':>10s} {'logodds(he-she)':>18s}")
for occ, phe, pshe, sc in rows_sorted:
    print(f"{occ:12s} {phe:10.6f} {pshe:10.6f} {sc:18.6f}")

occupation        P(he)     P(she)    logodds(he-she)
engineer       0.819799   0.052531           4.407373
CEO            0.850858   0.079585           4.189351
programmer     0.787653   0.111405           3.387301
janitor        0.750207   0.168626           2.695114
scientist      0.690605   0.141886           2.602661
secretary      0.689560   0.248867           1.902727
doctor         0.479885   0.327786           0.637713
assistant      0.538948   0.413901           0.503970
teacher        0.466574   0.376856           0.369010
nurse          0.187043   0.636520          -2.029633


#### Gender Stereotype Bias in Masked Language Modeling

We evaluated pronoun prediction probabilities for occupation templates:
    “The [occupation] said that [MASK] was tired.”

Results show strong asymmetry:
- Technical and high-status occupations (engineer, CEO, programmer) strongly favor “he”.
- Care-associated roles (nurse) strongly favor “she”.
- Log-odds differences exceed 4.0 in some cases.

This indicates that the pretrained masked language model encodes occupational gender stereotypes learned from large-scale training corpora.

This probe demonstrates measurable representation-level bias independent of downstream task fine-tuning.

## Key Findings

- Perturbation explanations more stable than gradients.
- Identity swaps affect explanation distributions even when predictions stable.
- MLM exhibits occupation-based gender priors.
- Robustness ≠ fairness.

## Reflection & Concept Check

### Key Takeaways

* **Perturbation vs Gradient**
  Ablation measures behavioral impact (finite removal).
  Gradients measure local sensitivity (infinitesimal change).
  Stability differences reflect geometric vs causal perspectives.

* **Robustness**
  Stable predictions do not guarantee stable explanations.
  Explanation distance (L2) quantifies reasoning shift under paraphrase.

* **Counterfactual Fairness**
  Removing identity tokens isolates indirect influence.
  If explanations shift but predictions don’t, bias may exist at the representation level.

* **MLM Bias Probe**
  Log-odds differences reveal pretrained occupational gender priors.
  Upstream bias can influence downstream classifiers.

## Quick Concept Check

**Q1. What does large explanation distance imply?**
Model reasoning shifts under small input changes.

**Q2. Does stable prediction imply fairness?**
No. Hidden representations may still encode bias.

**Q3. Why normalize explanation vectors?**
To compare structural shifts rather than scale differences.

**Q4. Why use counterfactual identity swaps?**
To test invariance of internal reasoning.

## Discussion Prompts

* Should explanations remain invariant under paraphrase?
* Can a model be robust yet biased?
* Which method is more trustworthy: gradients or perturbation?
* Is representational bias harmful even if outputs are stable?


## References

1. Nguyen, Duy, et al. "GrAInS: Gradient-based Attribution for Inference-Time Steering of LLMs and VLMs." arXiv preprint arXiv:2507.18043 (2025). 
2. Atmakuri, Shriya, et al. "Robustness of Explanation Methods for NLP Models." arXiv preprint arXiv:2206.12284 (2022).
3. Ross, Andrew Slavin, Michael C. Hughes, and Finale Doshi-Velez. "Right for the right reasons: Training differentiable models by constraining their explanations." arXiv preprint arXiv:1703.03717 (2017).
4. Kurita, Keita, et al. "Measuring bias in contextualized word representations." Proceedings of the first workshop on gender bias in natural language processing. 2019.